In [3]:
import pandas as pd
import os,json,re


In [4]:

os.chdir(r"C:\LLM\Python3.10\Code")
df = pd.read_excel("extracted_kbas 1.xlsx")
df.head()


,KBAID,Title,Content
0,3009016,You are Unable to Find The Desired Extension f...,Symptom\nYou have created few extension fields...
1,3009015,How to Populate Discount and Surcharge in Purc...,Symptom\nYou have created a purchase order and...
2,3008983,You are Unable to Find the Return to Supplier ...,Symptom\nYou have created a return to supplier...
3,3008820,Total Confirmed Duration field in the Resource...,Symptom\nYou observe Field 'Total Confirmed Du...
4,3008677,Unit of measure is different in purchase order...,Symptom\nYou notice that Unit of measure is no...


In [8]:
df_no_newlines = df.replace(r"[\r\n]+", " ", regex=True)
df_no_newlines.head()

,KBAID,Title,Content
0,3009016,You are Unable to Find The Desired Extension f...,Symptom You have created few extension fields ...
1,3009015,How to Populate Discount and Surcharge in Purc...,Symptom You have created a purchase order and ...
2,3008983,You are Unable to Find the Return to Supplier ...,Symptom You have created a return to supplier ...
3,3008820,Total Confirmed Duration field in the Resource...,Symptom You observe Field 'Total Confirmed Dur...
4,3008677,Unit of measure is different in purchase order...,Symptom You notice that Unit of measure is not...


In [11]:
section_names = ["Symptom", "Environment", "Reproducing the Issue", "Resolution"]
labels = "|".join(
    re.escape(name) for name in sorted(section_names, key=len, reverse=True)
)
heading_pattern = re.compile(
    rf"(?i)\b({labels})\b\s*:?\s*"
)

records = []

for _, row in df.iterrows():
    content = re.sub(r"\s+", " ", str(row["Content"])).strip()
    matches = list(heading_pattern.finditer(content))
    sections = {name: "" for name in section_names}

    for i, match in enumerate(matches):
        name = next(
            section for section in section_names
            if section.casefold() == match.group(1).casefold()
        )
        end = matches[i + 1].start() if i + 1 < len(matches) else len(content)
        sections[name] = content[match.end():end].strip()

    records.append({
        "KBAID": row["KBAID"],
        "TITLE": row["Title"],
        "CONTENT": content,
        **sections
    })

json_data = json.dumps(records, ensure_ascii=False, indent=2)

with open("kba_records.json", "w", encoding="utf-8") as f:
    f.write(json_data)

print(f"Saved {len(records)} records to kba_records.json")

Saved 11576 records to kba_records.json
